In [0]:
CREATE OR REPLACE VIEW fraud_alerts AS

WITH rule_flags AS (
    SELECT
        Transaction_ID,
        User_ID,
        Transaction_Amount,
        Number_of_Transactions_Last_24H,
        Previous_Fraudulent_Transactions,
        Fraudulent,

        CASE
            WHEN Transaction_Amount > 3000 THEN 1
            ELSE 0
        END AS large_amount_flag,

        CASE
            WHEN Number_of_Transactions_Last_24H > 10 THEN 1
            ELSE 0
        END AS high_activity_flag,

        CASE
            WHEN Previous_Fraudulent_Transactions > 0 THEN 1
            ELSE 0
        END AS previous_fraud_flag,

        CASE
            WHEN Transaction_Amount IS NULL THEN 1
            ELSE 0
        END AS missing_amount_flag

    FROM fraud_transactions_for_detection
),

rule_scores AS (
    SELECT
        *,
        large_amount_flag
            + high_activity_flag
            + previous_fraud_flag AS rules_triggered

    FROM rule_flags
)

SELECT
    *,

    CASE
        WHEN rules_triggered >= 2 THEN 1
        ELSE 0
    END AS predicted_fraud

FROM rule_scores;

SELECT
    Transaction_ID,
    Transaction_Amount,
    large_amount_flag,
    high_activity_flag,
    previous_fraud_flag,
    missing_amount_flag,
    rules_triggered,
    predicted_fraud,
    Fraudulent
FROM fraud_alerts
WHERE predicted_fraud = 1
ORDER BY rules_triggered DESC, Transaction_Amount DESC
LIMIT 20;

SELECT
    predicted_fraud,
    Fraudulent AS actual_fraud,
    COUNT(*) AS transaction_count
FROM fraud_alerts
GROUP BY predicted_fraud, Fraudulent
ORDER BY predicted_fraud, actual_fraud;